Silver to Gold Transformation

Objective
Create analytics-ready datasets from the curated Silver layer to support reporting and insights.

Gold Tables Created

earthquakes_gold_daily_summary
Provides daily earthquake statistics
Metrics include total count, average magnitude, and maximum magnitude
earthquakes_gold_by_region
Aggregates earthquake activity by region/location
Enables regional comparison and analysis
earthquakes_gold_magnitude_distribution
Categorizes earthquakes into magnitude ranges
Supports distribution and trend analysis

In [1]:
# --------------------------------------------------
# Gold Layer
#
# Purpose
# Create analytics-ready tables from the Silver layer.
#
# Gold Tables
# 1. earthquakes_gold_daily_summary
#    Daily earthquake statistics (count, avg magnitude, max magnitude)
#
# 2. earthquakes_gold_by_region
#    Earthquake activity grouped by region
#
# 3. earthquakes_gold_magnitude_distribution
#    Distribution of earthquakes by magnitude ranges
# --------------------------------------------------

from pyspark.sql.functions import col, avg, max, count, to_date

#1. earthquakes daily summary =====================>>>>>


# --------------------------------------------------
# Step 1 — Load Silver table
# --------------------------------------------------
df_silver = spark.read.table("earthquakes_silver")

row_count = df_silver.count()
print(f"Rows in earthquakes_silver: {row_count}")

display(df_silver.limit(2))


# --------------------------------------------------
# Step 2 — Extract event date (extract Date from UTC DateTime)
# --------------------------------------------------
df_event_date = df_silver.withColumn("event_date",to_date(col("event_time_utc")))

display(df_event_date.limit(2))


# --------------------------------------------------
# Step 3 — Aggregate daily earthquake statistics
# --------------------------------------------------
df_grouped_by_day = df_event_date.groupBy("event_date")

df_daily_summary = df_grouped_by_day.agg(
    count("*").alias("total_earthquakes"),
    avg("magnitude").alias("avg_magnitude"),
    max("magnitude").alias("max_magnitude")
)

display(df_daily_summary.sort("event_date", ascending=False).limit(15))


# --------------------------------------------------
# Step 4 — Save Gold table
# Overwrite is used because Gold is recomputed from Silver
# --------------------------------------------------
df_daily_summary.write.mode("overwrite").saveAsTable(
    "earthquakes_gold_daily_summary"
)

StatementMeta(, 96e680c5-fa2f-4f6f-ac48-461ddf093692, 3, Finished, Available, Finished, False)

Rows in earthquakes_silver: 10713


SynapseWidget(Synapse.DataFrame, 3a770e3a-c71d-4164-aff9-80b49614658f)

SynapseWidget(Synapse.DataFrame, a2f70c4a-804e-4dcc-a5d2-7b49dc301ba9)

SynapseWidget(Synapse.DataFrame, c4d7b510-dd9b-459e-99f0-a9809d7a1158)

In [2]:
#2. earthquakes by region =====================>>>>>

#Create earthquake counts by region (place)
df_grouped_by_region = df_silver.groupBy("place")

df_region_summary = df_grouped_by_region.agg(
    count("*").alias("total_earthquakes"),
    avg("magnitude").alias("avg_magnitude"),
    max("magnitude").alias("max_magnitude")
)

display(df_region_summary.sort("total_earthquakes", ascending=False).limit(15))
#Write to a table
df_region_summary.write.mode("overwrite").saveAsTable("earthquakes_gold_by_region")


StatementMeta(, 96e680c5-fa2f-4f6f-ac48-461ddf093692, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b27e51b8-c44f-449a-b85b-3657e1d0ffaf)

In [ ]:
df_silver.printSchema()

In [5]:
#3. earthquakes by magnitude bucket =====================>>>>>

from pyspark.sql.functions import when

df_with_mag_range = df_silver.withColumn(
    "magnitude_range",
    when(col("magnitude") < 2, "Micro (<2)")
    .when((col("magnitude") >= 2) & (col("magnitude") < 4), "Minor (2-4)")
    .when((col("magnitude") >= 4) & (col("magnitude") < 6), "Moderate (4-6)")
    .otherwise("Strong (6+)")
)

#group by magnitude range
df_grouped_mag = df_with_mag_range.groupBy("magnitude_range")

#Count earthquakes
df_mag_distribution = df_grouped_mag.agg(
    count("*").alias("total_earthquakes")
)

display(df_mag_distribution.sort("total_earthquakes", ascending=False))

df_mag_distribution.write.mode("overwrite").saveAsTable("earthquakes_gold_magnitude_distribution")

StatementMeta(, 96e680c5-fa2f-4f6f-ac48-461ddf093692, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 643ae5bd-0e1e-4e3e-98f4-c1756b4fc598)